# [16.4] TokenSHAP and TokenShapley - Exercises

Implement exact and sampled token-position Shapley attribution, then run the visible tests in each cell. The CUDA verification report for the trained finite token scorer is committed next to this notebook.

In [ ]:
import itertools
import math
import random
import sys
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path

import torch as t

chapter = "chapter16_shapley_attribution_baselines"
section = "part4_tokenshap_token_shapley"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part4_tokenshap_token_shapley.tests as tests
import part4_tokenshap_token_shapley.utils as utils

GT_TIER = "GT-0"
EXERCISE_ID = "16.4.tokenshap_and_tokenshapley"
EXPECTED_RUNTIME = "35-50 minutes for exercises; about 1-2 minutes for the CUDA preflight"
REQUIRES_GPU = True

TOKENS = ("The", "capital", "is", "Paris")
MASK_TOKEN = "[MASK]"
Coalition = frozenset[int]

In [ ]:
@dataclass(frozen=True)
class TokenShapleySamplingReport:
    tokens: tuple[str, ...]
    exact_values: t.Tensor
    sampled_values: t.Tensor
    max_abs_error: float
    top_token: str
    sampled_top_token: str
    rank_matches: bool
    approximates_exact: bool


@dataclass(frozen=True)
class TokenBaselineReport:
    full_score: float
    baseline_score: float
    total_delta: float
    shapley_sum: float
    efficiency_error: float
    satisfies_efficiency: bool

## Coalition Enumeration

In [ ]:
def all_coalitions(num_players: int) -> tuple[Coalition, ...]:
    raise NotImplementedError()


tests.test_all_coalitions_enumerates_complete_powerset(all_coalitions)

## Toy Token Game

In [ ]:
def keyword_interaction_token_score(
    tokens: Sequence[str],
    *,
    target_token: str = "Paris",
    context_token: str = "capital",
    target_weight: float = 1.0,
    interaction_weight: float = 2.0,
) -> float:
    raise NotImplementedError()


tests.test_keyword_interaction_token_score_is_target_context_game(
    keyword_interaction_token_score,
)

In [ ]:
def token_coalition_values(
    tokens: Sequence[str],
    score_fn: Callable[[tuple[str, ...]], float],
    *,
    mask_token: str = MASK_TOKEN,
) -> dict[Coalition, float]:
    raise NotImplementedError()


tests.test_token_coalition_values_masks_absent_positions(token_coalition_values)

## Exact Token Shapley

In [ ]:
def normalize_coalition_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> dict[Coalition, float]:
    values = {frozenset(key): float(value) for key, value in coalition_values.items()}
    expected = set(all_coalitions(num_players))
    missing = expected - set(values)
    if missing:
        raise ValueError(f"coalition value table is missing {len(missing)} coalitions.")
    return values


def exact_shapley_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
) -> t.Tensor:
    raise NotImplementedError()


def exact_token_shapley_values(
    tokens: Sequence[str],
    score_fn: Callable[[tuple[str, ...]], float],
    *,
    mask_token: str = MASK_TOKEN,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_exact_token_shapley_values_splits_context_target_interaction(
    exact_token_shapley_values,
)

In [ ]:
def token_baseline_report(
    tokens: Sequence[str],
    score_fn: Callable[[tuple[str, ...]], float],
    *,
    mask_token: str = MASK_TOKEN,
    tolerance: float = 1e-9,
) -> TokenBaselineReport:
    raise NotImplementedError()


tests.test_token_baseline_report_checks_efficiency(token_baseline_report)

## Sampled TokenSHAP

In [ ]:
def sampled_permutation_shapley_values(
    coalition_values: Mapping[Coalition | tuple[int, ...], float],
    *,
    num_players: int,
    num_samples: int,
    seed: int = 0,
) -> t.Tensor:
    raise NotImplementedError()


def sampled_token_shapley_values(
    tokens: Sequence[str],
    score_fn: Callable[[tuple[str, ...]], float],
    *,
    mask_token: str = MASK_TOKEN,
    num_samples: int = 256,
    seed: int = 0,
) -> t.Tensor:
    token_tuple = tuple(tokens)
    values = token_coalition_values(token_tuple, score_fn, mask_token=mask_token)
    return sampled_permutation_shapley_values(
        values,
        num_players=len(token_tuple),
        num_samples=num_samples,
        seed=seed,
    )


def token_shapley_sampling_report(
    tokens: Sequence[str],
    score_fn: Callable[[tuple[str, ...]], float],
    *,
    mask_token: str = MASK_TOKEN,
    num_samples: int = 256,
    seed: int = 0,
    tolerance: float = 0.15,
) -> TokenShapleySamplingReport:
    raise NotImplementedError()


tests.test_token_shapley_sampling_report_matches_exact_ranking(
    token_shapley_sampling_report,
)

## Smoke Contract

In [ ]:
def _tensor_report(report) -> dict:
    result = report.__dict__.copy()
    for key, value in list(result.items()):
        if hasattr(value, "tolist"):
            result[key] = value.tolist()
    return result


def token_coalition_smoke_test() -> dict:
    values = token_coalition_values(TOKENS, keyword_interaction_token_score)
    return {
        "empty": values[frozenset()],
        "target_only": values[frozenset({3})],
        "context_and_target": values[frozenset({1, 3})],
    }


def exact_token_shapley_smoke_test() -> dict:
    exact = exact_token_shapley_values(TOKENS, keyword_interaction_token_score)
    return {
        "tokens": list(TOKENS),
        "exact_values": exact.tolist(),
        "baseline": token_baseline_report(
            TOKENS,
            keyword_interaction_token_score,
        ).__dict__,
    }


def sampled_tokenshap_smoke_test() -> dict:
    report = token_shapley_sampling_report(
        TOKENS,
        keyword_interaction_token_score,
        num_samples=512,
        seed=0,
        tolerance=0.1,
    )
    return _tensor_report(report)


def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    return {
        "coalitions": token_coalition_smoke_test(),
        "exact": exact_token_shapley_smoke_test(),
        "sampled": sampled_tokenshap_smoke_test(),
    }


tests.test_notebook_contract(run_smoke_test)

## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
